<a href="https://colab.research.google.com/github/spaceacer/NASA-SP8000-series-Archive/blob/main/NASA_8000_downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install requests beautifulsoup4 pymupdf tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 54.3 MB/s eta 0:00:00


In [2]:
from __future__ import annotations
import csv
import re
import shutil
import time
import unicodedata
from pathlib import Path
from urllib.parse import parse_qs, quote, unquote, urljoin, urlparse
import pymupdf
import requests
from bs4 import BeautifulSoup
from tqdm import tqdm
NAKKA_URL = 'https://www.nakka-rocketry.net/sp8000.html'
OUTPUT_DIR = Path('NASA_SP8000')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = OUTPUT_DIR / 'manifest.csv'
TIMEOUT = 60
DELAY = 0.25
SESSION = requests.Session()
SESSION.headers.update({'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/152 Safari/537.36', 'Accept': '*/*'})

def clean_filename(text: str, max_len: int=170) -> str:
    """
    Convert:
        Flutter, Buzz, and Divergence

    into:
        flutter_buzz_and_divergence
    """
    text = unicodedata.normalize('NFKD', text)
    text = text.encode('ascii', 'ignore').decode('ascii')
    text = text.lower()
    text = text.replace('&', ' and ')
    text = re.sub('[–—-]+', ' ', text)
    text = re.sub("['’]", '', text)
    text = re.sub('[/\\\\:;,.\\(\\)\\[\\]\\{\\}]+', ' ', text)
    text = re.sub('[^a-z0-9\\s]', '', text)
    text = re.sub('\\s+', '_', text).strip('_')
    return text[:max_len].rstrip('_')

def extract_sp_number(text: str | None) -> str | None:
    if not text:
        return None
    match = re.search('\\b(?:NASA\\s*[/\\-]?\\s*)?SP[\\s\\-_/]*(8\\d{3})\\b', text, flags=re.I)
    if match:
        return match.group(1)
    return None

def extract_current_ntrs_id(text: str | None) -> str | None:
    """
    Current NTRS citation IDs generally look like:

        19690014753
        19710021390

    We deliberately require a long numeric identifier so that
    Wayback timestamps are not mistaken for NTRS IDs.
    """
    if not text:
        return None
    patterns = ['ntrs\\.nasa\\.gov/citations/(\\d{10,14})', 'ntrs\\.nasa\\.gov/api/citations/(\\d{10,14})', '/downloads/(\\d{10,14})\\.pdf']
    for pattern in patterns:
        m = re.search(pattern, text, flags=re.I)
        if m:
            return m.group(1)
    return None

def extract_casi_id(text: str | None) -> str | None:
    """
    Legacy TRS pages use identifiers such as:

        70N71604
        N70-71604
    """
    if not text:
        return None
    patterns = ['CASI\\s+Document\\s+ID\\s+Number\\s*:?\\s*([A-Z0-9\\-]+)', '\\b(\\d{2}N\\d{5})\\b', '\\b(N\\d{2}-\\d{5})\\b']
    for pattern in patterns:
        m = re.search(pattern, text, flags=re.I)
        if m:
            return m.group(1).upper()
    return None

def is_wayback_url(url: str) -> bool:
    return 'web.archive.org' in urlparse(url).netloc.lower()

def is_current_ntrs_url(url: str) -> bool:
    host = urlparse(url).netloc.lower()
    return 'ntrs.nasa.gov' in host and 'web.archive.org' not in host

def looks_like_pdf_file(path: Path) -> bool:
    if not path.exists():
        return False
    if path.stat().st_size < 1000:
        return False
    try:
        with path.open('rb') as f:
            return f.read(5) == b'%PDF-'
    except OSError:
        return False

def validate_pdf(path: Path) -> bool:
    """
    Much stronger than merely checking extension.
    """
    if not looks_like_pdf_file(path):
        return False
    try:
        doc = pymupdf.open(path)
        valid = len(doc) >= 1 and (not doc.needs_pass)
        doc.close()
        return valid
    except Exception:
        return False

def save_stream_as_pdf(response, destination: Path) -> bool:
    tmp = destination.with_suffix('.part')
    tmp.unlink(missing_ok=True)
    try:
        with tmp.open('wb') as f:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        if validate_pdf(tmp):
            tmp.replace(destination)
            return True
    finally:
        if tmp.exists():
            tmp.unlink(missing_ok=True)
    return False

def get(url: str, *, stream: bool=False, allow_redirects: bool=True):
    last_error = None
    for attempt in range(4):
        try:
            r = SESSION.get(url, timeout=TIMEOUT, stream=stream, allow_redirects=allow_redirects)
            if r.status_code in (429, 500, 502, 503, 504):
                time.sleep(2 ** attempt)
                continue
            return r
        except requests.RequestException as exc:
            last_error = exc
            time.sleep(2 ** attempt)
    if last_error:
        raise last_error
    raise RuntimeError(f'Unable to fetch {url}')

def scrape_nakka():
    print('Reading Nakka SP-8000 index...')
    response = get(NAKKA_URL)
    response.raise_for_status()
    soup = BeautifulSoup(response.text, 'html.parser')
    records = {}
    order = []
    for a in soup.find_all('a', href=True):
        href = urljoin(NAKKA_URL, a['href'])
        row = a.find_parent('tr') or a.find_parent('p') or a.find_parent('li') or a.parent
        context = row.get_text(' ', strip=True) if row else a.get_text(' ', strip=True)
        sp = extract_sp_number(context)
        if not sp:
            sp = extract_sp_number(a.get_text(' ', strip=True))
        if not sp:
            continue
        lower_href = href.lower()
        interesting = 'nasa.gov' in lower_href or 'web.archive.org' in lower_href
        if not interesting:
            continue
        if sp not in records:
            records[sp] = {'sp': sp, 'nakka_text': context, 'nakka_url': href}
            order.append(sp)
        else:
            current = records[sp]['nakka_url']
            new_score = link_quality(href)
            old_score = link_quality(current)
            if new_score > old_score:
                records[sp]['nakka_url'] = href
                records[sp]['nakka_text'] = context
    result = [records[sp] for sp in order]
    result.sort(key=lambda x: int(x['sp']))
    print(f'Found {len(result)} SP entries.')
    return result

def link_quality(url: str) -> int:
    low = url.lower()
    if 'ntrs.nasa.gov/citations/' in low:
        return 100
    if 'web.archive.org' in low and '/archive/' in low:
        return 90
    if 'web.archive.org' in low:
        return 60
    if 'nasa.gov' in low:
        return 50
    return 0
WAYBACK_PATTERN = re.compile('^https?://web\\.archive\\.org/web/(\\d+)(?:[a-z_]+)?/(https?.+)$', flags=re.I)

def parse_wayback_url(url: str):
    """
    Returns:
        timestamp
        original_url
    """
    m = WAYBACK_PATTERN.match(url)
    if not m:
        return (None, None)
    timestamp = m.group(1)
    original_url = unquote(m.group(2))
    original_url = re.sub('^(https?):/(?!/)', '\\1://', original_url, flags=re.I)
    return (timestamp, original_url)

def make_wayback_raw_url(original_url: str, timestamp: str | None=None):
    """
    id_ tells Wayback to replay the original payload
    rather than its HTML wrapper.
    """
    if timestamp:
        return f'https://web.archive.org/web/{timestamp}id_/{original_url}'
    return f'https://web.archive.org/web/0id_/{original_url}'

def absolutize_wayback_link(base_wayback_url: str, link: str):
    """
    Handles all of these:

        relative links
        absolute archived links
        original NASA links
    """
    link = link.strip()
    if not link:
        return None
    if link.startswith('//'):
        link = 'https:' + link
    if link.startswith(('http://', 'https://')):
        if 'web.archive.org' in urlparse(link).netloc:
            return link
        timestamp, _ = parse_wayback_url(base_wayback_url)
        return make_wayback_raw_url(link, timestamp)
    timestamp, original_base = parse_wayback_url(base_wayback_url)
    if original_base:
        original_pdf = urljoin(original_base, link)
        return make_wayback_raw_url(original_pdf, timestamp)
    return urljoin(base_wayback_url, link)

def get_wayback_record(url: str):
    """
    Opens an archived TRS/MTRS record and obtains:

        page title
        CASI ID
        archived PDF URLs
    """
    response = get(url)
    response.raise_for_status()
    html = response.text
    soup = BeautifulSoup(html, 'html.parser')
    page_text = soup.get_text('\n', strip=True)
    title = None
    for tag_name in ('h1', 'h2', 'h3'):
        tag = soup.find(tag_name)
        if tag:
            candidate = tag.get_text(' ', strip=True)
            if candidate and 'wayback' not in candidate.lower() and ('marshall technical' not in candidate.lower()):
                title = candidate
                break
    if not title and soup.title:
        candidate = soup.title.get_text(' ', strip=True)
        if candidate:
            title = candidate
    casi_id = extract_casi_id(page_text)
    pdf_urls = []
    for a in soup.find_all('a', href=True):
        href = a['href']
        label = a.get_text(' ', strip=True).lower()
        if '.pdf' not in href.lower() and 'pdf' not in label:
            continue
        candidate = absolutize_wayback_link(url, href)
        if candidate and candidate not in pdf_urls:
            pdf_urls.append(candidate)
        if candidate and is_wayback_url(candidate):
            ts, original = parse_wayback_url(candidate)
            if original:
                raw = make_wayback_raw_url(original, ts)
                if raw not in pdf_urls:
                    pdf_urls.append(raw)
    return {'title': title, 'casi_id': casi_id, 'pdf_urls': pdf_urls, 'page_text': page_text}

def walk_json(obj):
    if isinstance(obj, dict):
        yield obj
        for value in obj.values():
            yield from walk_json(value)
    elif isinstance(obj, list):
        for value in obj:
            yield from walk_json(value)

def get_current_ntrs_record(ntrs_id: str):
    url = f'https://ntrs.nasa.gov/api/citations/{ntrs_id}'
    response = get(url)
    if response.status_code != 200:
        return None
    try:
        return response.json()
    except Exception:
        return None

def find_urls_in_ntrs_record(record):
    urls = []
    for obj in walk_json(record):
        for value in obj.values():
            if isinstance(value, str):
                low = value.lower()
                if '.pdf' in low or '/downloads/' in low:
                    if value.startswith('/'):
                        value = 'https://ntrs.nasa.gov' + value
                    if value.startswith('http') and value not in urls:
                        urls.append(value)
    return urls

def metadata_title(record):
    if not record:
        return None
    value = record.get('title')
    if isinstance(value, str):
        if value.strip():
            return value.strip()
    for obj in walk_json(record):
        value = obj.get('title')
        if isinstance(value, str) and value.strip():
            return value.strip()
    return None

def get_ntrs_pdf_candidates(ntrs_id: str, record):
    urls = find_urls_in_ntrs_record(record)
    fallbacks = [f'https://ntrs.nasa.gov/api/citations/{ntrs_id}/downloads/{ntrs_id}.pdf', f'https://ntrs.nasa.gov/api/citations/{ntrs_id}/downloads/{ntrs_id}.pdf?attachment=true']
    for url in fallbacks:
        if url not in urls:
            urls.append(url)
    return urls

def download_first_working_pdf(candidates, destination: Path):
    errors = []
    for url in candidates:
        try:
            response = get(url, stream=True)
            if response.status_code != 200:
                errors.append(f'{response.status_code}: {url}')
                continue
            if save_stream_as_pdf(response, destination):
                return (url, errors)
            errors.append(f'Not PDF: {url}')
        except Exception as exc:
            errors.append(f'{type(exc).__name__}: {url} :: {exc}')
    return (None, errors)
BOILERPLATE_PATTERNS = ['national aeronautics', 'space administration', 'nasa\\s+space\\s+vehicle', 'space vehicle design criteria', 'scientific and technical information', 'washington,\\s*d\\.?c', 'nasa\\s+sp[\\s\\-]*8\\d{3}']

def extract_pdf_title(pdf_path: Path):
    """
    Primary objective:
    get the title printed on the first page.

    Uses font sizes and positioning.
    """
    try:
        doc = pymupdf.open(pdf_path)
        if len(doc) == 0:
            doc.close()
            return None
        page = doc[0]
        page_height = page.rect.height
        info = page.get_text('dict')
        spans = []
        for block in info.get('blocks', []):
            for line in block.get('lines', []):
                for span in line.get('spans', []):
                    text = span.get('text', '').strip()
                    if not text:
                        continue
                    bbox = span.get('bbox', [0, 0, 0, 0])
                    spans.append({'text': text, 'size': float(span.get('size', 0)), 'x': float(bbox[0]), 'y': float(bbox[1])})
        doc.close()
        if not spans:
            return None
        spans = [s for s in spans if s['y'] <= page_height * 0.75]
        filtered = []
        for s in spans:
            lower = s['text'].lower()
            if any((re.search(pattern, lower) for pattern in BOILERPLATE_PATTERNS)):
                continue
            filtered.append(s)
        if not filtered:
            return None
        meaningful_sizes = [s['size'] for s in filtered if s['size'] >= 8]
        if not meaningful_sizes:
            return None
        max_size = max(meaningful_sizes)
        title_spans = [s for s in filtered if s['size'] >= max_size * 0.8]
        title_spans.sort(key=lambda s: (round(s['y'], 1), s['x']))
        title = ' '.join((s['text'] for s in title_spans))
        title = re.sub('\\s+', ' ', title).strip()
        title = re.sub('^(?:NASA\\s*)?SP[\\s\\-/]*8\\d{3}\\s*', '', title, flags=re.I)
        title = title.strip(' -:;,.')
        if len(title) < 4:
            return None
        return title
    except Exception:
        return None

def pdf_contains_expected_sp(pdf_path: Path, sp: str):
    """
    Soft validation. We do not reject a PDF merely because OCR/text
    extraction failed, but this catches obvious wrong-document cases.
    """
    try:
        doc = pymupdf.open(pdf_path)
        pages = min(len(doc), 3)
        text = ''
        for i in range(pages):
            text += '\n' + doc[i].get_text('text')
        doc.close()
        normalized = re.sub('\\s+', '', text.upper())
        candidates = [f'SP-{sp}', f'SP{sp}', f'NASA/SP-{sp}']
        candidates = [re.sub('\\s+', '', x.upper()) for x in candidates]
        return any((candidate in normalized for candidate in candidates))
    except Exception:
        return None

def choose_identifier(ntrs_id: str | None, casi_id: str | None):
    """
    Preferred:
        current NTRS numeric ID

    Fallback:
        historical CASI ID

    We DO NOT invent an NTRS number.
    """
    if ntrs_id:
        return (ntrs_id, 'NTRS')
    if casi_id:
        return (casi_id, 'CASI')
    return ('unknown_id', 'UNKNOWN')

def process_current_ntrs(sp: str, source_url: str, temp_path: Path):
    ntrs_id = extract_current_ntrs_id(source_url)
    response = get(source_url)
    final_url = response.url
    if not ntrs_id:
        ntrs_id = extract_current_ntrs_id(final_url)
    if not ntrs_id:
        return {'success': False, 'error': 'Current NTRS link but no numeric citation ID found'}
    record = get_current_ntrs_record(ntrs_id)
    if not record:
        return {'success': False, 'ntrs_id': ntrs_id, 'error': 'NTRS citation API record unavailable'}
    candidates = get_ntrs_pdf_candidates(ntrs_id, record)
    used_url, errors = download_first_working_pdf(candidates, temp_path)
    if not used_url:
        return {'success': False, 'ntrs_id': ntrs_id, 'error': 'No working PDF URL | ' + ' || '.join(errors)}
    return {'success': True, 'ntrs_id': ntrs_id, 'casi_id': None, 'metadata_title': metadata_title(record), 'download_url': used_url, 'source_type': 'current_ntrs', 'resolved_url': final_url}

def process_wayback(sp: str, source_url: str, temp_path: Path):
    record = get_wayback_record(source_url)
    candidates = record['pdf_urls']
    timestamp, original = parse_wayback_url(source_url)
    if original:
        original_dir = original.rstrip('/') + '/01/' + f'sp{sp}.pdf'
        candidate = make_wayback_raw_url(original_dir, timestamp)
        if candidate not in candidates:
            candidates.append(candidate)
    used_url, errors = download_first_working_pdf(candidates, temp_path)
    if not used_url:
        return {'success': False, 'casi_id': record['casi_id'], 'metadata_title': record['title'], 'error': 'Archived record found but PDF download failed | ' + ' || '.join(errors)}
    return {'success': True, 'ntrs_id': None, 'casi_id': record['casi_id'], 'metadata_title': record['title'], 'download_url': used_url, 'source_type': 'wayback_trs', 'resolved_url': source_url}

def process_record(sp: str, source_url: str, temp_path: Path):
    if is_wayback_url(source_url):
        return process_wayback(sp, source_url, temp_path)
    if is_current_ntrs_url(source_url):
        return process_current_ntrs(sp, source_url, temp_path)
    response = get(source_url)
    final_url = response.url
    if is_wayback_url(final_url):
        return process_wayback(sp, final_url, temp_path)
    if is_current_ntrs_url(final_url):
        return process_current_ntrs(sp, final_url, temp_path)
    return {'success': False, 'error': 'Unsupported final URL: ' + final_url}

def main():
    entries = scrape_nakka()
    rows = []
    for entry in tqdm(entries, desc='Downloading SP-8000'):
        sp = entry['sp']
        source_url = entry['nakka_url']
        temp_path = OUTPUT_DIR / f'_temp_sp{sp}.pdf'
        temp_path.unlink(missing_ok=True)
        print(f'\nSP-{sp}')
        print(f'  Source: {source_url}')
        try:
            result = process_record(sp, source_url, temp_path)
            if not result.get('success'):
                error = result.get('error', 'Unknown error')
                print(f'  FAILED: {error}')
                rows.append({'sp': f'SP-{sp}', 'identifier': '', 'identifier_type': '', 'ntrs_id': result.get('ntrs_id', '') or '', 'casi_id': result.get('casi_id', '') or '', 'title': '', 'filename': '', 'status': 'FAILED', 'source_type': '', 'nakka_url': source_url, 'resolved_url': '', 'download_url': '', 'sp_validation': '', 'error': error})
                continue
            pdf_title = extract_pdf_title(temp_path)
            metadata_title_value = result.get('metadata_title')
            title = pdf_title or metadata_title_value or f'NASA SP {sp}'
            ntrs_id = result.get('ntrs_id')
            casi_id = result.get('casi_id')
            identifier, identifier_type = choose_identifier(ntrs_id, casi_id)
            safe_title = clean_filename(title)
            filename = f'sp{sp}_{identifier}_{safe_title}.pdf'
            final_path = OUTPUT_DIR / filename
            sp_check = pdf_contains_expected_sp(temp_path, sp)
            if final_path.exists():
                temp_path.unlink(missing_ok=True)
                status = 'ALREADY_EXISTS'
            else:
                temp_path.replace(final_path)
                status = 'OK'
            print(f'  ID: {identifier} ({identifier_type})')
            print(f'  Title: {title}')
            print(f'  Saved: {filename}')
            if sp_check is False:
                print('  WARNING: SP number was not found in extractable PDF text.')
            rows.append({'sp': f'SP-{sp}', 'identifier': identifier, 'identifier_type': identifier_type, 'ntrs_id': ntrs_id or '', 'casi_id': casi_id or '', 'title': title, 'filename': filename, 'status': status, 'source_type': result.get('source_type', ''), 'nakka_url': source_url, 'resolved_url': result.get('resolved_url', ''), 'download_url': result.get('download_url', ''), 'sp_validation': sp_check, 'error': ''})
        except KeyboardInterrupt:
            raise
        except Exception as exc:
            temp_path.unlink(missing_ok=True)
            error = f'{type(exc).__name__}: {exc}'
            print(f'  ERROR: {error}')
            rows.append({'sp': f'SP-{sp}', 'identifier': '', 'identifier_type': '', 'ntrs_id': '', 'casi_id': '', 'title': '', 'filename': '', 'status': 'ERROR', 'source_type': '', 'nakka_url': source_url, 'resolved_url': '', 'download_url': '', 'sp_validation': '', 'error': error})
        time.sleep(DELAY)
    fieldnames = ['sp', 'identifier', 'identifier_type', 'ntrs_id', 'casi_id', 'title', 'filename', 'status', 'source_type', 'nakka_url', 'resolved_url', 'download_url', 'sp_validation', 'error']
    with MANIFEST_PATH.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    successful = sum((row['status'] in ('OK', 'ALREADY_EXISTS') for row in rows))
    failed = len(rows) - successful
    print()
    print('=' * 72)
    print('FINISHED')
    print('=' * 72)
    print(f'Entries     : {len(rows)}')
    print(f'Successful  : {successful}')
    print(f'Failed      : {failed}')
    print(f'Manifest    : {MANIFEST_PATH}')
    archive = shutil.make_archive('NASA_SP8000', 'zip', OUTPUT_DIR)
    print(f'ZIP         : {archive}')
if __name__ == '__main__':
    main()


Reading Nakka SP-8000 index...
Found 112 SP entries.



SP-8001
  Source: https://ntrs.nasa.gov/citations/19710010998
  ID: 19710010998 (NTRS)
  Title: BUFFETINGDURING ATMOSPHERIC ASCENT
  Saved: sp8001_19710010998_buffetingduring_atmospheric_ascent.pdf



SP-8002
  Source: https://web.archive.org/web/20100528141531/http:/trs.nis.nasa.gov/archive/00000001/
  ID: 66N85660 (CASI)
  Title: Flight-loads measurements during launch and exit.
  Saved: sp8002_66N85660_flight_loads_measurements_during_launch_and_exit.pdf



SP-8003
  Source: https://web.archive.org/web/20100528141607/http:/trs.nis.nasa.gov/archive/00000003/
  ID: 70N71604 (CASI)
  Title: Flutter, buzz, and divergence
  Saved: sp8003_70N71604_flutter_buzz_and_divergence.pdf



SP-8004
  Source: https://ntrs.nasa.gov/citations/19730003205
  ID: 19730003205 (NTRS)
  Title: PANEL FLUTTER
  Saved: sp8004_19730003205_panel_flutter.pdf


...


SP-8059
  Source: https://ntrs.nasa.gov/citations/19710016722
  ID: 19710016722 (NTRS)
  Title: I ATTITUDE SPACECRAFT CONTROL DURING THRUSTING MANEUVERS
  Saved: sp8059_19710016722_i_attitude_spacecraft_control_during_thrusting_maneuvers.pdf



SP-8060
  Source: https://ntrs.nasa.gov/citations/19710018690
  ID: 19710018690 (NTRS)
  Title: COMPARTMENT VENTING
  Saved: sp8060_19710018690_compartment_venting.pdf



SP-8061
  Source: https://ntrs.nasa.gov/citations/19710019353
  ID: 19710019353 (NTRS)
  Title: C A UMBILICALS INTERACTION WITH STAND LAUNCH AND
  Saved: sp8061_19710019353_c_a_umbilicals_interaction_with_stand_launch_and.pdf



SP-8062
  Source: https://ntrs.nasa.gov/citations/19710021703


KeyboardInterrupt: 